# A1 — `C.U40k.can.e5 → Ut`: canonical SMILES, 5 epochs

The reference configuration of the A-series: plain canonical SMILES on both sides, no
augmentation, no representation change. Everything A2-A4 add is measured against this row.

Five epochs rather than the three used on the ORD pools: every CompoundT5 run so far ended with
`eval_loss` still falling at the last step, and this pool is smaller (39,667 rows) than any of
them, so the epoch budget was the binding constraint rather than the data.


**Benchmark.** `bisectgroup/USPTO_50K`, the canonical 40008/5001/5007 split used across the
retrosynthesis literature. Training rows come from its `train` partition, the test set from its
`test` partition (5003 unique products; 82 rows whose product also occurs in `test` were dropped
from the training side, so the overlap is exactly zero). The base `sagawa/CompoundT5` was
pretrained by span-MLM over 24M ZINC20 *molecules* and has never seen a reaction, so nothing in
this benchmark can leak through pretraining -- unlike the ORD line, where
`ReactionT5v2-retrosynthesis` was pretrained on ~1.5M ORD reactions and the ORD test is drawn
from the same database.

**Reference points (literature, same split):** R-SMILES 56.3% top-1 / 86.2% top-5 exact;
RetroKNN 57.2% top-1.


**Data:** `kuzmenkooleh/retro-planner-uspto50k-canonical` (39,667 train + 4,986 val).

**Cost:** ~1 h 50 min training + ~35 min evaluation.

**Before running:** Settings -> **Internet** on, **GPU T4 x2** on. Run as **Save & Run All (Commit)**.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):

    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os

if not os.path.isdir("retro-planner"):

    !git clone https://github.com/oleh-kuzmenko/retro-planner.git

%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob

train_file = next(glob.iglob("/kaggle/input/**/reactants_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/reactants_val.jsonl", recursive=True))
for path in (train_file, val_file):
    print(path, sum(1 for _ in open(path)), "rows")

base_model = "sagawa/CompoundT5"
learning_rate = 5e-4
num_train_epochs = 5
output_dir = "/kaggle/working/model1_compoundt5_uspto_can"
time_budget_minutes = 150

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

!torchrun --nproc_per_node=2 scripts/train_reactant_model_ord.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model1_work \
    --no-augment \
    --learning-rate {learning_rate} \
    --num-train-epochs {num_train_epochs} \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done; tail of log:")
!tail -5 "{log_path}"

In [ ]:
# The vocabulary repair must have fired (CompoundT5 ships a 221-token ZINC vocabulary with
# no `.` separator). If this line is absent the run trained on <unk>-corrupted targets.
!grep -E "new character token|Train examples" "{log_path}"

import json
from transformers import AutoTokenizer

saved_tokenizer = AutoTokenizer.from_pretrained(f"{output_dir}/final")
embedding_rows = json.load(open(f"{output_dir}/final/config.json"))["vocab_size"]
print("tokenizer length:", len(saved_tokenizer), "| embedding rows:", embedding_rows)
assert len(saved_tokenizer) == embedding_rows, "tokenizer and embedding matrix disagree"

In [ ]:
# Whether eval_loss was still falling at the end decides what the next run changes:
# still falling -> the epoch budget binds; flattened early -> the data or the recipe does.
import json

state = json.load(open(f"{output_dir}/latest_checkpoint/trainer_state.json"))
points = [(h["epoch"], h["eval_loss"]) for h in state["log_history"] if "eval_loss" in h]
for epoch, loss in points[:: max(1, len(points) // 10)]:
    print(f"  epoch {epoch:5.2f}  eval_loss {loss:.4f}")
print("  last:", points[-1], "| best:", state.get("best_metric"))

In [ ]:
# Comparison set: 1000 records, a strict prefix of the full 5003-record canonical test
# split (same seed), so numbers here and on the full set are drawn from one distribution.
model_dir = f"{output_dir}/final"
!python scripts/models/run_reactiont5_topk.py \
    --input "data/v2_uspto_test_holdout_1000.json" --t5-model "{model_dir}" \
    --num-beams 10 --device cuda \
    --output "/kaggle/working/A1_can_e5_uspto1000_topk.json"

In [ ]:
import json
data = json.load(open("/kaggle/working/A1_can_e5_uspto1000_topk.json"))
print(json.dumps(data["summary"], indent=2))